In [1]:
%pip install pymysql

  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd

# 讀取 csv 檔案
df = pd.read_csv('./Sample-Superstore.csv', encoding='latin1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
import pymysql
from configparser import ConfigParser
from create_table import create_superstore_tables

# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    # 建立資料庫
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS superstore")
    cursor.execute(f"SHOW DATABASES")
    dbs = cursor.fetchall()
    print(dbs)

    # 建立資料表
    create_superstore_tables('superstore')

[{'Database': 'chapter2'}, {'Database': 'chapter3'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'superstore'}, {'Database': 'sys'}, {'Database': 'testdb'}, {'Database': 'transaction_test'}, {'Database': 'world'}]
{'Tables_in_superstore': 'customers'}
{'Tables_in_superstore': 'orderdetails'}
{'Tables_in_superstore': 'orders'}
{'Tables_in_superstore': 'products'}


In [20]:
# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database='superstore'
)

In [17]:
import pandas as pd

# 讀取 CSV 檔案
df = pd.read_csv('./Sample-Superstore.csv', encoding='latin1')
# df.head()
print(df.columns)

# 提取客戶欄位的資料
customers = df[['Customer ID', 'Customer Name', 'Segment']]
# 去除重複的客戶資料
customers = customers.drop_duplicates()

customers

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
5,BH-11710,Brosina Hoffman,Consumer
12,AA-10480,Andrew Allen,Consumer
...,...,...,...
8666,CJ-11875,Carl Jackson,Corporate
9209,RS-19870,Roy Skaria,Home Office
9399,SC-20845,Sung Chung,Consumer
9441,RE-19405,Ricardo Emerson,Consumer


Customers

In [ ]:
import pandas as pd

# 讀取 CSV 檔案
df = pd.read_csv('./Sample-Superstore.csv', encoding='latin1')
# df.head()
print(df.columns)

# 提取客戶欄位的資料
customers = df[['Customer ID', 'Customer Name', 'Segment']]
# 去除重複的客戶資料
customers = customers.drop_duplicates()

# 轉成 list 以便後續寫入
customers_list = customers.values.tolist()
print(customers_list)

# 寫入 Customers 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO customers (customer_id, customer_name, segment)
        VALUES (%s, %s, %s);
    """
    # cursor.executemany(sql, customers_list) # 執行過就先註解起來，避免重複寫入
    
    cursor.execute('SELECT * FROM customers;')
    result = cursor.fetchall()
    print(result)

    connection.commit()

[['CG-12520', 'Claire Gute', 'Consumer'], ['DV-13045', 'Darrin Van Huff', 'Corporate'], ['SO-20335', "Sean O'Donnell", 'Consumer'], ['BH-11710', 'Brosina Hoffman', 'Consumer'], ['AA-10480', 'Andrew Allen', 'Consumer'], ['IM-15070', 'Irene Maddox', 'Consumer'], ['HP-14815', 'Harold Pawlan', 'Home Office'], ['PK-19075', 'Pete Kriz', 'Consumer'], ['AG-10270', 'Alejandro Grove', 'Consumer'], ['ZD-21925', 'Zuschuss Donatelli', 'Consumer'], ['KB-16585', 'Ken Black', 'Corporate'], ['SF-20065', 'Sandra Flanagan', 'Consumer'], ['EB-13870', 'Emily Burns', 'Consumer'], ['EH-13945', 'Eric Hoffmann', 'Consumer'], ['TB-21520', 'Tracy Blumstein', 'Consumer'], ['MA-17560', 'Matt Abelman', 'Home Office'], ['GH-14485', 'Gene Hale', 'Corporate'], ['SN-20710', 'Steve Nguyen', 'Home Office'], ['LC-16930', 'Linda Cazamias', 'Corporate'], ['RA-19885', 'Ruben Ausman', 'Corporate'], ['ES-14080', 'Erin Smith', 'Corporate'], ['ON-18715', 'Odella Nelson', 'Corporate'], ['PO-18865', "Patrick O'Donnell", 'Consumer'

Orders 

In [33]:
from datetime import datetime

# 取得 df 中的訂單資料
print(df.columns)
orders_df = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID']]


# orders_df.head()
# 將日期格式轉換為 datetime.date
orders_df['Order Date'] = pd.to_datetime(orders_df['Order Date'], format='%m/%d/%Y').dt.date
orders_df['Ship Date'] = pd.to_datetime(orders_df['Ship Date'], format='%m/%d/%Y').dt.date


orders_df = orders_df.drop_duplicates()

# orders_df
# 轉成 list 以便後續寫入
orders_list = orders_df.values.tolist()

# 寫入 Orders 資料表
with connection.cursor() as cursor:
    
    sql = """
        INSERT INTO orders (order_id, order_date, ship_date, ship_mode, customer_id) 
        VALUES (%s, %s, %s, %s, %s)
    """
    cursor.executemany(sql, orders_list)
    connection.commit()


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


C:\Users\student\AppData\Local\Temp\ipykernel_16392\636102154.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_df['Order Date'] = pd.to_datetime(orders_df['Order Date'], format='%m/%d/%Y').dt.date
C:\Users\student\AppData\Local\Temp\ipykernel_16392\636102154.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_df['Ship Date'] = pd.to_datetime(orders_df['Ship Date'], format='%m/%d/%Y').dt.date


Products

In [35]:
# 取得 df 中的產品資料
print(df.columns)
products_df = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']]

# 轉成 list 以便後續寫入
products_df = products_df.drop_duplicates()

products_list = products_df.values.tolist()

# 寫入 Products 資料表
with connection.cursor() as cursor:
    
    sql = """
        INSERT INTO products (product_id, product_name, category, sub_category)
        VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(sql, products_list)
    connection.commit()

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


OrderDetails

In [38]:
# 取得 df 中的訂單明細資料
print(df.columns)
orderdetails_df = df[['Row ID', 'Order ID', 'Product ID', 'Sales', 'Quantity', 'Discount', 'Profit']]

# 轉成 list 以便後續寫入
orderdetails_list = orderdetails_df.values.tolist()

# 寫入 OrderDetails 資料表
with connection.cursor() as cursor:
    
    sql = """
        INSERT INTO orderdetails (row_id, order_id, product_id, sales, quantity, discount, profit)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(sql, orderdetails_list)
    connection.commit()



Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')
